# 05 — Pattern Discovery & Validation

**Goal:** Map the 31 feature clusters (FC1–FC31) from NB04 to failure events, validate their relevance, and prepare a neutral summary for expert verification with Wiener Linien.

**Note:** Cluster labels are intentionally neutral (FC1–FC31). Subsystem interpretation is left to domain experts.

**Pipeline:** NB01 → NB02 → NB03 → **NB04** (feature clustering) → **NB05** (pattern discovery & validation)

In [1]:
import sys, os, gc, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

ROOT = os.path.abspath("..")
OUT = os.path.join(ROOT, "outputs")
ARTIFACT_DIR = os.path.join(OUT, "nb04_artifacts")
NB05_OUT = os.path.join(OUT, "nb05_artifacts")
os.makedirs(NB05_OUT, exist_ok=True)

# Load NB04 artifacts
all_sup = pd.read_parquet(os.path.join(ARTIFACT_DIR, "all_sup.parquet"))
all_sup["window_end"] = pd.to_datetime(all_sup["window_end"])

anom_activations = pd.read_parquet(os.path.join(ARTIFACT_DIR, "anom_activations.parquet"))
anom_meta = pd.read_parquet(os.path.join(ARTIFACT_DIR, "anom_meta.parquet"))
anom_orig_idx = np.load(os.path.join(ARTIFACT_DIR, "anom_orig_idx.npy"))

with open(os.path.join(ARTIFACT_DIR, "active_feats.json")) as f:
    active_feats = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "feature_cols.json")) as f:
    feature_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "heuristic_a.json")) as f:
    fc_clusters = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "constants.json")) as f:
    constants = json.load(f)

Z_THRESH = constants["Z_THRESH"]
MIN_ACTIVATED = constants["MIN_ACTIVATED"]
SPLIT_DATE = pd.Timestamp(constants["SPLIT_DATE"])

clusters = {int(k): v for k, v in fc_clusters.items()}
anom_activations.index = pd.Index(anom_orig_idx)
anom_meta.index = pd.Index(anom_orig_idx)
fc_ids_sorted = sorted(clusters.keys())

print(f"Loaded: {len(all_sup):,} windows, {len(feature_cols)} features")
print(f"Anomalous windows: {len(anom_activations):,}")
print(f"Feature clusters (FCs): {len(clusters)}")
print(f"Features in clusters: {sum(len(v) for v in clusters.values())}")

Loaded: 64,522 windows, 266 features
Anomalous windows: 12,022
Feature clusters (FCs): 31
Features in clusters: 130


## 1. FC Overview: Size, Sensors & Activation Rates

For each FC: how many features, which sensors, and how often it fires in anomalous windows.

In [2]:
rows = []
for fc_id in fc_ids_sorted:
    feats = clusters[fc_id]
    fc_active = anom_activations[feats].any(axis=1)
    act_rate = fc_active.mean()
    fc_labels = all_sup.loc[anom_activations.index[fc_active], "label"]
    fail_rate = fc_labels.mean() if len(fc_labels) > 0 else 0

    # Extract unique sensor names (strip aggregation suffix)
    sensors = set()
    for f in feats:
        parts = f.rsplit("_", 1)
        if parts[-1] in ("min", "max", "mean", "std", "flips"):
            sensors.add(parts[0])
        else:
            sensors.add(f)

    rows.append({
        "FC": f"FC{fc_id}", "n_features": len(feats),
        "n_sensors": len(sensors),
        "activation_rate": act_rate, "n_active_windows": int(fc_active.sum()),
        "failure_rate": fail_rate,
        "sensors": ", ".join(sorted(sensors)[:3]) + ("..." if len(sensors) > 3 else "")
    })

fc_summary = pd.DataFrame(rows)
print(fc_summary[["FC", "n_features", "n_sensors", "activation_rate", "failure_rate", "sensors"]].to_string(index=False))

  FC  n_features  n_sensors  activation_rate  failure_rate                                                                                                                sensors
 FC1           4          4         0.056480      0.106038                      CW1_SPRING_BRAKE_ACTIVE_BOGIE1, CW1_SPRING_BRAKE_ACTIVE_BOGIE2, CW2_SPRING_BRAKE_ACTIVE_BOGIE1...
 FC2           1          1         0.040759      0.077551                                                                                                 CW1_COMPRESSOR_RUNNING
 FC3           1          1         0.028614      0.154070                                                                                                 CW2_COMPRESSOR_RUNNING
 FC4          10         10         0.410747      0.077157          CW1_BRAKE_CYLINDER_PRESSURE_BOGIE2, CW2_BRAKE_CYLINDER_PRESSURE_BOGIE2, MW1_BRAKE_CYLINDER_PRESSURE_BOGIE1...
 FC5           3          3         0.156047      0.048507 MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1, MW2_PROPORT

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fc_ids = [f"FC{i}" for i in fc_ids_sorted]
sizes = [len(clusters[i]) for i in fc_ids_sorted]

# 1) FC sizes
ax = axes[0]
colors = ["#2ecc71" if s > 1 else "#95a5a6" for s in sizes]
ax.barh(fc_ids, sizes, color=colors)
ax.set_xlabel("Number of features")
ax.set_title("FC Sizes (green = multi-feature)")
ax.invert_yaxis()

# 2) Activation rate
ax = axes[1]
act_rates = fc_summary["activation_rate"].values
ax.barh(fc_ids, act_rates * 100, color="#3498db")
ax.set_xlabel("Activation rate (% of anomalous windows)")
ax.set_title("FC Activation Rate")
ax.invert_yaxis()

# 3) Failure rate per FC
ax = axes[2]
fail_rates = fc_summary["failure_rate"].values
baseline = all_sup.loc[anom_activations.index, "label"].mean()
bar_colors = ["#e74c3c" if fr > baseline * 2 else "#f39c12" if fr > baseline else "#95a5a6" for fr in fail_rates]
ax.barh(fc_ids, fail_rates * 100, color=bar_colors)
ax.axvline(baseline * 100, color="black", ls="--", lw=1, label=f"Baseline ({baseline*100:.1f}%)")
ax.set_xlabel("Failure rate (%)")
ax.set_title("Failure Rate per FC (red = >2x baseline)")
ax.invert_yaxis()
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(NB05_OUT, "fc_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved fc_overview.png")

Saved fc_overview.png


C:\Users\Emırhan\AppData\Local\Temp\ipykernel_3688\4001664734.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Full sensor list per FC — for expert review
print("Feature Clusters — Full Sensor Lists")
print("=" * 100)
for fc_id in fc_ids_sorted:
    feats = clusters[fc_id]
    print(f"\nFC{fc_id} ({len(feats)} features):")
    for f in sorted(feats):
        print(f"  {f}")

Feature Clusters — Full Sensor Lists

FC1 (4 features):
  CW1_SPRING_BRAKE_ACTIVE_BOGIE1_flips
  CW1_SPRING_BRAKE_ACTIVE_BOGIE2_flips
  CW2_SPRING_BRAKE_ACTIVE_BOGIE1_flips
  CW2_SPRING_BRAKE_ACTIVE_BOGIE2_flips

FC2 (1 features):
  CW1_COMPRESSOR_RUNNING_flips

FC3 (1 features):
  CW2_COMPRESSOR_RUNNING_flips

FC4 (10 features):
  CW1_BRAKE_CYLINDER_PRESSURE_BOGIE2_max
  CW2_BRAKE_CYLINDER_PRESSURE_BOGIE2_max
  MW1_BRAKE_CYLINDER_PRESSURE_BOGIE1_max
  MW1_BRAKE_CYLINDER_PRESSURE_BOGIE2_max
  MW2_BRAKE_CYLINDER_PRESSURE_BOGIE1_max
  MW2_BRAKE_CYLINDER_PRESSURE_BOGIE2_max
  MW3_BRAKE_CYLINDER_PRESSURE_BOGIE1_max
  MW3_BRAKE_CYLINDER_PRESSURE_BOGIE2_max
  MW4_BRAKE_CYLINDER_PRESSURE_BOGIE1_max
  MW4_BRAKE_CYLINDER_PRESSURE_BOGIE2_max

FC5 (3 features):
  MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1_max
  MW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE1_max
  MW4_PROPORTIONAL_VALVE_PRESSURE_BOGIE1_max

FC6 (4 features):
  MW1_PNEUMATIC_BRAKING_FORCE_BOGIE1_max
  MW1_PNEUMATIC_BRAKING_FORCE_BOGIE2_max
  M

## 2. Link FCs to Failure Events

Load raw failure event timestamps and map each anomalous window to the nearest failure event. Cross-tabulate: which FCs fire before/during failures?

In [5]:
import glob

TS_COL = "TIMESTAMP"
FAIL_COL = "TRAIN_IS_IN_FAILURE"
FAIL_TYPE = "TRAIN_FAILURE_TYPE"
HORIZON_S = 22326  # ~6.2 hours prediction horizon

all_files = sorted(
    glob.glob(os.path.join(ROOT, "train", "**", "*.parquet"), recursive=True) +
    glob.glob(os.path.join(ROOT, "test", "**", "*.parquet"), recursive=True)
)
chunks = []
for f in all_files:
    df = pd.read_parquet(f, columns=[TS_COL, FAIL_COL, FAIL_TYPE])
    chunks.append(df)
raw_meta = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
raw_meta[TS_COL] = pd.to_datetime(raw_meta[TS_COL])
raw_meta.sort_values(TS_COL, inplace=True)

mask = raw_meta[FAIL_COL].astype(bool)
grp = (mask != mask.shift()).cumsum()
fail_events = (
    raw_meta[mask].groupby(grp[mask])
    .agg(start=(TS_COL, "min"), end=(TS_COL, "max"),
         event_type=(FAIL_TYPE, "first"), n_rows=(TS_COL, "count"))
    .reset_index(drop=True)
)

merged = []
for _, row in fail_events.iterrows():
    if merged and row["event_type"] == merged[-1]["event_type"] and \
       (row["start"] - merged[-1]["end"]).total_seconds() < 3600:
        merged[-1]["end"] = row["end"]
        merged[-1]["n_rows"] += row["n_rows"]
    else:
        merged.append(dict(row))
fail_events = pd.DataFrame(merged)
fail_events["duration_min"] = (fail_events["end"] - fail_events["start"]).dt.total_seconds() / 60
fail_events["period"] = fail_events["start"].apply(lambda x: "train" if x < SPLIT_DATE else "test")
fail_events.index += 1
fail_events.index.name = "event_id"
del raw_meta; gc.collect()

print(f"Failure events: {len(fail_events)}")
print(f"  Train: {(fail_events['period']=='train').sum()}, Test: {(fail_events['period']=='test').sum()}")
print(f"\nEvent types:")
print(fail_events["event_type"].value_counts().to_string())

Failure events: 21
  Train: 10, Test: 11

Event types:
event_type
Brake System Failure         13
Compressor Module Failure     6
Leveling System Failure       2


In [6]:
def find_nearest_failure(window_end, fail_events, horizon_s):
    horizon_end = window_end + pd.Timedelta(seconds=horizon_s)
    for idx, ev in fail_events.iterrows():
        if ev["start"] >= window_end and ev["start"] <= horizon_end:
            return idx, ev["event_type"], "precursor"
        if ev["start"] <= window_end and ev["end"] >= window_end:
            return idx, ev["event_type"], "during"
    return None, None, "none"

anom_meta = anom_meta.copy()
anom_meta["fail_event_id"] = None
anom_meta["fail_event_type"] = None
anom_meta["fail_relation"] = "none"

print("Mapping anomalous windows to failure events...")
for i, row in anom_meta.iterrows():
    eid, etype, rel = find_nearest_failure(row["window_end"], fail_events, HORIZON_S)
    anom_meta.at[i, "fail_event_id"] = eid
    anom_meta.at[i, "fail_event_type"] = etype
    anom_meta.at[i, "fail_relation"] = rel

print(f"\nAnomalous window - failure relation:")
print(anom_meta["fail_relation"].value_counts().to_string())

Mapping anomalous windows to failure events...



Anomalous window - failure relation:
fail_relation
none         11298
during         403
precursor      321


In [7]:
print("FC to Failure Event Mapping")
print("=" * 100)

fc_fail_matrix = []
for fc_id in fc_ids_sorted:
    feats = clusters[fc_id]
    fc_active_mask = anom_activations[feats].any(axis=1)
    fc_meta = anom_meta.loc[fc_active_mask]
    n_total = fc_active_mask.sum()
    n_precursor = (fc_meta["fail_relation"] == "precursor").sum()
    n_during = (fc_meta["fail_relation"] == "during").sum()
    n_none = (fc_meta["fail_relation"] == "none").sum()
    covered_events = fc_meta[fc_meta["fail_relation"].isin(["precursor", "during"])]["fail_event_id"].dropna().unique()
    fc_fail_matrix.append({
        "FC": f"FC{fc_id}", "n_windows": n_total,
        "precursor": n_precursor, "during": n_during, "none": n_none,
        "fail_pct": (n_precursor + n_during) / n_total * 100 if n_total > 0 else 0,
        "events_covered": len(covered_events), "event_ids": list(covered_events)
    })
    if n_precursor + n_during > 0:
        print(f"\nFC{fc_id} ({len(feats)} feats): {n_total} windows, "
              f"{n_precursor} precursor, {n_during} during, "
              f"{n_none} unrelated | covers {len(covered_events)} events")

fc_fail_df = pd.DataFrame(fc_fail_matrix)
print(f"\n\nFCs with failure association: {(fc_fail_df['fail_pct'] > 0).sum()} / {len(fc_fail_df)}")

FC to Failure Event Mapping

FC1 (4 feats): 679 windows, 41 precursor, 31 during, 607 unrelated | covers 16 events

FC2 (1 feats): 490 windows, 11 precursor, 27 during, 452 unrelated | covers 15 events

FC3 (1 feats): 344 windows, 16 precursor, 37 during, 291 unrelated | covers 16 events

FC4 (10 feats): 4938 windows, 174 precursor, 207 during, 4557 unrelated | covers 16 events

FC5 (3 feats): 1876 windows, 60 precursor, 31 during, 1785 unrelated | covers 11 events

FC6 (4 feats): 1097 windows, 63 precursor, 49 during, 985 unrelated | covers 11 events

FC7 (4 feats): 259 windows, 14 precursor, 17 during, 228 unrelated | covers 9 events

FC8 (3 feats): 701 windows, 18 precursor, 62 during, 621 unrelated | covers 12 events

FC9 (1 feats): 326 windows, 7 precursor, 11 during, 308 unrelated | covers 10 events

FC10 (4 feats): 2352 windows, 132 precursor, 74 during, 2146 unrelated | covers 11 events

FC11 (6 feats): 847 windows, 34 precursor, 68 during, 745 unrelated | covers 13 events

FC1

## 3. FC Validation Scorecard

For each FC: failure association and event coverage. Classification: **KEEP** (relevant for predictive maintenance) vs **REVIEW** (needs expert input to determine relevance).

In [8]:
baseline_fail = all_sup.loc[anom_activations.index, "label"].mean()
print(f"Baseline failure rate among anomalous windows: {baseline_fail*100:.1f}%")

scorecard = []
for fc_id in fc_ids_sorted:
    feats = clusters[fc_id]
    fc_row = fc_fail_df[fc_fail_df["FC"] == f"FC{fc_id}"].iloc[0]
    fail_rate = fc_summary[fc_summary["FC"] == f"FC{fc_id}"]["failure_rate"].values[0]
    events_covered = fc_row["events_covered"]
    fail_score = "HIGH" if fail_rate > baseline_fail * 2 else "MED" if fail_rate > baseline_fail else "LOW"
    verdict = "KEEP" if fail_score in ("HIGH", "MED") and events_covered > 0 else "REVIEW"
    scorecard.append({
        "FC": f"FC{fc_id}", "n_feats": len(feats),
        "fail_rate": f"{fail_rate*100:.1f}%",
        "fail_assoc": fail_score, "events": events_covered, "verdict": verdict
    })

scorecard_df = pd.DataFrame(scorecard)
print("\n" + scorecard_df.to_string(index=False))
print(f"\nKEEP: {(scorecard_df['verdict']=='KEEP').sum()}, REVIEW: {(scorecard_df['verdict']=='REVIEW').sum()}")

Baseline failure rate among anomalous windows: 6.0%

  FC  n_feats fail_rate fail_assoc  events verdict
 FC1        4     10.6%        MED      16    KEEP
 FC2        1      7.8%        MED      15    KEEP
 FC3        1     15.4%       HIGH      16    KEEP
 FC4       10      7.7%        MED      16    KEEP
 FC5        3      4.9%        LOW      11  REVIEW
 FC6        4     10.2%        MED      11    KEEP
 FC7        4     12.0%        MED       9    KEEP
 FC8        3     11.4%        MED      12    KEEP
 FC9        1      5.5%        LOW      10  REVIEW
FC10        4      8.8%        MED      11    KEEP
FC11        6     12.0%        MED      13    KEEP
FC12        6      3.4%        LOW      13  REVIEW
FC13        1      7.9%        MED       8    KEEP
FC14        2      7.3%        MED       8    KEEP
FC15        3      8.0%        MED      13    KEEP
FC16        2      8.8%        MED      11    KEEP
FC17        1     15.2%       HIGH       4    KEEP
FC18        3     28.8%      

## 4. Timeline: FC Activations vs Failure Events

Show when each FC fires over the full observation period, overlaid with failure events.

In [9]:
fig, ax = plt.subplots(figsize=(24, 10))
cmap = plt.cm.tab20(np.linspace(0, 1, len(fc_ids_sorted)))

for row_idx, fc_id in enumerate(fc_ids_sorted):
    feats = clusters[fc_id]
    fc_active_mask = anom_activations[feats].any(axis=1)
    fc_times = anom_meta.loc[fc_active_mask, "window_end"]
    ax.scatter(fc_times, [row_idx] * len(fc_times), s=2, alpha=0.4, color=cmap[row_idx])

for _, ev in fail_events.iterrows():
    ax.axvspan(ev["start"], ev["end"], color="red", alpha=0.15)

ax.axvline(SPLIT_DATE, color="black", ls="--", lw=1.5, label="Train/Test split")
ax.set_yticks(range(len(fc_ids_sorted)))
ax.set_yticklabels([f"FC{i}" for i in fc_ids_sorted], fontsize=8)
ax.set_xlabel("Time")
ax.set_title("FC Activation Timeline (red shading = failure events)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(NB05_OUT, "fc_timeline.png"), dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Emırhan\AppData\Local\Temp\ipykernel_3688\308711983.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Deep Dive: Key FCs

Detailed analysis of the FCs with the strongest failure association from NB04:
- **FC28**: fires before failures, variance ramps up (potential early warning)
- **FC24**: highest failure-day activation increase (+28.7pp)
- **FC19**: fires during failures, not before (detection-only pattern)

Sensor names are shown as-is — expert interpretation needed.

In [10]:
def deep_dive_fc(fc_id, n_top=5):
    feats = clusters[fc_id]
    fc_active = anom_activations[feats].any(axis=1)
    fc_meta_local = anom_meta.loc[fc_active]
    freq = anom_activations.loc[fc_active, feats].mean().sort_values(ascending=False)
    top_feats = freq.head(min(n_top, len(feats)))
    print(f"\n{'='*80}")
    print(f"FC{fc_id}: {len(feats)} features, {fc_active.sum()} active windows")
    print(f"Failure relation: {fc_meta_local['fail_relation'].value_counts().to_dict()}")
    print(f"\nTop features (activation rate):")
    for feat, rate in top_feats.items():
        print(f"  {feat}: {rate:.1%}")
    n_plots = len(top_feats)
    fig, axes = plt.subplots(n_plots, 1, figsize=(20, 3 * n_plots), sharex=True)
    if n_plots == 1: axes = [axes]
    for ax, feat in zip(axes, top_feats.index):
        if feat in all_sup.columns:
            daily = all_sup.groupby(all_sup["window_end"].dt.date)[feat].agg(["mean", "std"])
            daily.index = pd.to_datetime(daily.index)
            ax.plot(daily.index, daily["mean"], color="steelblue", lw=1, label="Daily mean")
            ax.fill_between(daily.index, daily["mean"] - daily["std"], daily["mean"] + daily["std"],
                          alpha=0.2, color="steelblue")
            for _, ev in fail_events.iterrows():
                ax.axvspan(ev["start"], ev["end"], color="red", alpha=0.15)
            ax.set_ylabel(feat[:30], fontsize=8)
    axes[0].set_title(f"FC{fc_id} - Top Feature Daily Trends (red = failure events)")
    plt.tight_layout()
    plt.savefig(os.path.join(NB05_OUT, f"fc{fc_id}_deep_dive.png"), dpi=150, bbox_inches="tight")
    plt.show()

for fc_id in [28, 24, 19]:
    if fc_id in clusters:
        deep_dive_fc(fc_id)


FC28: 1 features, 3925 active windows
Failure relation: {'none': 3527, 'during': 319, 'precursor': 79}

Top features (activation rate):
  MW4_LOAD_SIGNAL_min: 100.0%


C:\Users\Emırhan\AppData\Local\Temp\ipykernel_3688\3281310476.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



FC24: 10 features, 2626 active windows
Failure relation: {'none': 2275, 'during': 237, 'precursor': 114}

Top features (activation rate):
  MW3_PROPORTIONAL_VALVE_PRESSURE_BOGIE2_max: 62.4%
  MW3_SPRING_BRAKE_PRESSURE_BOGIE2_max: 52.0%
  MW3_LOAD_SIGNAL_mean: 39.8%
  MW3_PNEUMATIC_BRAKING_FORCE_BOGIE2_max: 35.6%
  MW3_LOAD_SIGNAL_min: 35.2%


C:\Users\Emırhan\AppData\Local\Temp\ipykernel_3688\3281310476.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



FC19: 7 features, 316 active windows
Failure relation: {'none': 176, 'during': 102, 'precursor': 38}

Top features (activation rate):
  MW4_SPRING_BRAKE_PRESSURE_BOGIE1_min: 63.0%
  MW4_SPRING_BRAKE_PRESSURE_BOGIE2_min: 60.4%
  MW2_SPRING_BRAKE_PRESSURE_BOGIE1_min: 51.6%
  MW3_SPRING_BRAKE_PRESSURE_BOGIE1_min: 47.2%
  MW2_SPRING_BRAKE_PRESSURE_BOGIE2_min: 45.9%


C:\Users\Emırhan\AppData\Local\Temp\ipykernel_3688\3281310476.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Expert Verification Summary

Neutral summary for the Wiener Linien meeting. Clusters are presented by their sensor content — **no subsystem labels assigned**. Experts should:
1. Confirm or reject each FC's physical relevance
2. Identify which FCs correspond to known failure modes
3. Prioritize FCs for the predictive model

In [11]:
print("=" * 90)
print("ANOMALY PATTERN DISCOVERY - SUMMARY FOR EXPERT VERIFICATION")
print("=" * 90)

print(f"\nDataset: {len(all_sup):,} windows ({(all_sup['split']=='train').sum():,} train, {(all_sup['split']=='test').sum():,} test)")
print(f"Anomalous windows: {len(anom_activations):,} ({len(anom_activations)/len(all_sup)*100:.1f}%)")
print(f"Feature clusters: {len(clusters)} (FC1-FC{max(clusters.keys())})")
print(f"Total features in clusters: {sum(len(v) for v in clusters.values())}")
print(f"Failure events: {len(fail_events)}")

print(f"\n--- Scorecard ---")
keep = scorecard_df[scorecard_df["verdict"] == "KEEP"]
print(f"KEEP: {len(keep)} FCs (failure-associated)")
for _, row in keep.iterrows():
    print(f"  {row['FC']}: {row['fail_rate']} failure rate, {row['events']} events covered")

review = scorecard_df[scorecard_df["verdict"] == "REVIEW"]
print(f"\nREVIEW: {len(review)} FCs (need expert input)")

print(f"\n--- Observations (not conclusions) ---")
print("1. FC28 fires BEFORE failures with variance ramp-up — potential early warning signal")
print("2. FC24 shows the highest activation increase on failure days (+28.7pp)")
print("3. FC19 fires DURING failures, not before — detection pattern, not prediction")
print("4. Most failure-associated anomalies appear in standing/low-speed windows")

print(f"\n--- Questions for Experts ---")
print("Q1: Which FCs correspond to known failure modes?")
print("Q2: Are any FCs capturing normal operational patterns (false positives)?")
print("Q3: Which FCs should be prioritized for the predictive model?")
print("Q4: Do the sensor groupings within each FC make physical sense?")
print("Q5: What subsystem does each FC belong to?")

ANOMALY PATTERN DISCOVERY - SUMMARY FOR EXPERT VERIFICATION

Dataset: 64,522 windows (41,375 train, 23,147 test)
Anomalous windows: 12,022 (18.6%)
Feature clusters: 31 (FC1-FC31)
Total features in clusters: 130
Failure events: 21

--- Scorecard ---
KEEP: 27 FCs (failure-associated)
  FC1: 10.6% failure rate, 16 events covered
  FC2: 7.8% failure rate, 15 events covered
  FC3: 15.4% failure rate, 16 events covered
  FC4: 7.7% failure rate, 16 events covered
  FC6: 10.2% failure rate, 11 events covered
  FC7: 12.0% failure rate, 9 events covered
  FC8: 11.4% failure rate, 12 events covered
  FC10: 8.8% failure rate, 11 events covered
  FC11: 12.0% failure rate, 13 events covered
  FC13: 7.9% failure rate, 8 events covered
  FC14: 7.3% failure rate, 8 events covered
  FC15: 8.0% failure rate, 13 events covered
  FC16: 8.8% failure rate, 11 events covered
  FC17: 15.2% failure rate, 4 events covered
  FC18: 28.8% failure rate, 15 events covered
  FC19: 44.3% failure rate, 15 events covered

In [12]:
scorecard_df.to_csv(os.path.join(NB05_OUT, "fc_scorecard.csv"), index=False)
fc_fail_df.to_csv(os.path.join(NB05_OUT, "fc_failure_mapping.csv"), index=False)
fc_summary.to_csv(os.path.join(NB05_OUT, "fc_summary.csv"), index=False)
anom_meta.to_parquet(os.path.join(NB05_OUT, "anom_meta_enriched.parquet"))
fail_events.to_parquet(os.path.join(NB05_OUT, "fail_events.parquet"))

# Save full FC sensor lists for expert review
fc_sensor_list = {}
for fc_id in fc_ids_sorted:
    fc_sensor_list[str(fc_id)] = sorted(clusters[fc_id])
with open(os.path.join(NB05_OUT, "fc_sensor_lists.json"), "w") as f:
    json.dump(fc_sensor_list, f, indent=2)

print(f"Saved NB05 artifacts to {NB05_OUT}:")
for f in sorted(os.listdir(NB05_OUT)):
    sz = os.path.getsize(os.path.join(NB05_OUT, f))
    print(f"  {f}: {sz:,} bytes")
print("\nNB05 complete. Ready for expert verification meeting.")

Saved NB05 artifacts to C:\Users\Emırhan\Desktop\MASTER THESIS\Kurtulus-thesis\outputs\nb05_artifacts:
  anom_meta_enriched.parquet: 204,314 bytes
  fail_events.parquet: 5,109 bytes
  fc19_deep_dive.png: 1,055,008 bytes
  fc24_deep_dive.png: 967,934 bytes
  fc28_deep_dive.png: 259,927 bytes
  fc_failure_mapping.csv: 2,849 bytes
  fc_overview.png: 98,091 bytes
  fc_scorecard.csv: 841 bytes
  fc_sensor_lists.json: 6,006 bytes
  fc_subsystem_map.png: 157,800 bytes
  fc_summary.csv: 3,675 bytes
  fc_timeline.png: 293,293 bytes
  subsystem_feature_groups.json: 6,379 bytes

NB05 complete. Ready for expert verification meeting.


## Traceability: Pipeline Position

**NB01 > NB02 > NB03 > NB04 > NB05**

### What this notebook provides:
- FC-to-failure event mapping (which FCs fire near which failures)
- Validation scorecard (KEEP vs REVIEW for each FC)
- Neutral cluster presentation with full sensor lists for expert review
- Deep dive into key FCs (FC28, FC24, FC19)
- Expert verification questions

### Outputs:
- `outputs/nb05_artifacts/fc_scorecard.csv`
- `outputs/nb05_artifacts/fc_failure_mapping.csv`
- `outputs/nb05_artifacts/fc_sensor_lists.json`
- `outputs/nb05_artifacts/anom_meta_enriched.parquet`
- `outputs/nb05_artifacts/fail_events.parquet`
- Plots: `fc_overview.png`, `fc_timeline.png`, `fc*_deep_dive.png`